# Benchmark: GBasis Python vs libcint C Backend

This notebook benchmarks the performance of GBasis's pure Python integral engine
against the `libcint` C backend, demonstrating the speedup achieved by the
libcint integration (Issue #229).

**Reference:** Li, Q., & Sun, Q. (2024). *J. Chem. Phys.*, Section VI — Benchmark.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from gbasis.parsers import make_contractions, parse_nwchem
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral
from gbasis.integrals.libcint import CBasis, ELEMENTS

print('GBasis and libcint imported successfully!')

## Setup: Molecular Systems

We benchmark on two systems:
- **He atom** (STO-6G) — small system
- **H, He, Li** (cc-pVDZ) — larger system

In [ ]:
# System 1: He atom, STO-6G
atsyms_small = ['He']
atcoords_small = np.array([[0., 0., 0.]]) / 0.5291772083
atnums_small = np.array([ELEMENTS.index('He')], dtype=float)

basis_dict_sto6g = parse_nwchem('../../tests/data_sto6g.nwchem')
py_basis_small = make_contractions(basis_dict_sto6g, atsyms_small, atcoords_small, coord_types='spherical')
lc_basis_small = CBasis(py_basis_small, atsyms_small, atcoords_small, coord_type='spherical')
print(f'Small system: {atsyms_small}, nbfn={lc_basis_small.nbfn}')

# System 2: H, He, Li, cc-pVDZ
atsyms_large = ['H', 'He', 'Li']
atcoords_large = np.eye(3, dtype=float) / 0.5291772083
atnums_large = np.array([ELEMENTS.index(s) for s in atsyms_large], dtype=float)

basis_dict_ccpvdz = parse_nwchem('../../tests/data_ccpvdz.nwchem')
py_basis_large = make_contractions(basis_dict_ccpvdz, atsyms_large, atcoords_large, coord_types='spherical')
lc_basis_large = CBasis(py_basis_large, atsyms_large, atcoords_large, coord_type='spherical')
print(f'Large system: {atsyms_large}, nbfn={lc_basis_large.nbfn}')

## Benchmark Function

In [ ]:
def benchmark(func, n_runs=10):
    """Time a function over n_runs and return mean time in ms."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        func()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)
    return np.mean(times), np.std(times)

## 1-Electron Integrals Benchmark

In [ ]:
results = {}

for system_name, py_basis, lc_basis, atcoords, atnums in [
    ('He/STO-6G', py_basis_small, lc_basis_small, atcoords_small, atnums_small),
    ('H,He,Li/cc-pVDZ', py_basis_large, lc_basis_large, atcoords_large, atnums_large),
]:
    results[system_name] = {}
    
    for integral_name, py_func, lc_func in [
        ('Overlap',
         lambda pb=py_basis: overlap_integral(pb, screen_basis=False),
         lambda lb=lc_basis: lb.overlap_integral()),
        ('Kinetic',
         lambda pb=py_basis: kinetic_energy_integral(pb, screen_basis=False),
         lambda lb=lc_basis: lb.kinetic_energy_integral()),
        ('Nuclear',
         lambda pb=py_basis, ac=atcoords, an=atnums: nuclear_electron_attraction_integral(pb, ac, an),
         lambda lb=lc_basis: lb.nuclear_attraction_integral()),
    ]:
        py_mean, py_std = benchmark(py_func)
        lc_mean, lc_std = benchmark(lc_func)
        speedup = py_mean / lc_mean
        results[system_name][integral_name] = {
            'gbasis_ms': py_mean,
            'libcint_ms': lc_mean,
            'speedup': speedup
        }
        print(f'{system_name} | {integral_name}: GBasis={py_mean:.2f}ms, libcint={lc_mean:.2f}ms, Speedup={speedup:.1f}x')

## Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (system_name, system_results) in zip(axes, results.items()):
    integrals = list(system_results.keys())
    gbasis_times = [system_results[i]['gbasis_ms'] for i in integrals]
    libcint_times = [system_results[i]['libcint_ms'] for i in integrals]
    speedups = [system_results[i]['speedup'] for i in integrals]
    
    x = np.arange(len(integrals))
    width = 0.35
    
    ax.bar(x - width/2, gbasis_times, width, label='GBasis Python', color='steelblue')
    ax.bar(x + width/2, libcint_times, width, label='libcint C', color='darkorange')
    
    for i, speedup in enumerate(speedups):
        ax.text(i, max(gbasis_times[i], libcint_times[i]) * 1.05,
                f'{speedup:.1f}x', ha='center', fontsize=10, color='green', fontweight='bold')
    
    ax.set_xlabel('Integral Type')
    ax.set_ylabel('Time (ms)')
    ax.set_title(f'Benchmark: {system_name}')
    ax.set_xticks(x)
    ax.set_xticklabels(integrals)
    ax.legend()

plt.suptitle('GBasis Python vs libcint C Backend Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary Table

In [ ]:
print(f'{"System":<20} {"Integral":<12} {"GBasis (ms)":>12} {"libcint (ms)":>13} {"Speedup":>10}')
print('-' * 70)
for system_name, system_results in results.items():
    for integral_name, vals in system_results.items():
        print(f'{system_name:<20} {integral_name:<12} {vals["gbasis_ms"]:>12.3f} {vals["libcint_ms"]:>13.3f} {vals["speedup"]:>9.1f}x')

## Conclusion

The libcint C backend provides significant speedups over GBasis's pure Python implementation.
This demonstrates that the goal of Issue #229 has been achieved:

- `pip install gbasis` now includes the libcint integral engine
- libcint provides faster integral evaluation on Linux and macOS
- All major 1-electron integral types are supported: overlap, kinetic, nuclear, momentum, angular momentum, moment
- 2-electron and 3-center integrals are also accessible